In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download('stopwords')
nltk.download('wordnet')

# Data Science Process

> CRISP-DM CRoss Industry Standard Process for Data Mining

![alt](../assets/Screenshot%202026-06-25%20at%2012.20.14.png)

# Step 1: Business Understanding

# Step 2: Data Understanding

Python-Setup mit UV und den notwendigen Abhängigkeiten eingerichtet.

> pyproject.toml als Dokumentation für die genutzten Pakete

```toml
[project]
name = "dhbw-neuekonzepte-portfolio"
version = "0.1.0"
description = "Add your description here"
readme = "README.md"
requires-python = ">=3.12"
dependencies = [
    "ipykernel>=7.2.0",
    "polars>=1.40.1",
    "pandas>=2.0.0",
    "matplotlib>=3.7.0",
    "seaborn>=0.13.0",
    "pyarrow>=14.0.0",
    "scikit-learn>=1.9.0",
    "nltk>=3.9.4",
]
```

### Datensatz

https://www.kaggle.com/datasets/pashupatigupta/emotion-detection-from-text


### SMART Ziele

> Spezifisch, Messbar, Attraktiv, Realistisch, Terminiert

**Forschungsfrage:** Lassen sich Emotionsklassen anhand von Kurznachrichten zuverlässig und automatisiert modellieren und vorhersagen?

Ziele:
1. Den gewählten Datensatz analysieren und nutzbar aufbereiten um die Modellvorhersagen zu ermöglichen 
2. Ziel 2
3. Ziel 3
4. Ziel 4

### Datensatz einlesen

In [ ]:
df = pd.read_csv('../data/csv/tweet_emotion_dataset.csv')   # Datensatz in CSV Format einlesen
df = df.drop(columns=['tweet_id'])                          # Drop 'tweet_id' column
df.columns = ['emotion', 'content']                         # Rename columns
df

In [ ]:
df.info()

Der gewählte Datensatz enthält 40.000 Zeilen sowie 3 Spalten. Spalte 1 (tweet_id) wird nicht benötigt und wurde daher entfernt. Spalte 2 enhält die Emotionsklasse zu der in Spalte 3 enthaltenten Kurznachricht (content).

*Ist der Datensatz intuitiv veständlich?*

In [ ]:
df.describe()

Beschreibung von df.describe()

+ 163 Dopplungen in Content
+ Visualisierung der Zielvariablen

### Visualisierung der Zielvariablen

In [ ]:
values = df["emotion"].value_counts()
labels = values.index
colors = plt.cm.tab10.colors

pct_labels = [f"{l} ({v/values.sum()*100:.1f}%)" for l, v in zip(labels, values)]

fig, (ax_bar, ax_pie) = plt.subplots(1, 2, figsize=(16, 6))

bars = ax_bar.bar(labels, values, color=colors[:len(labels)])
ax_bar.set_title("Verteilung der Zielvariable 'emotion'", fontsize=14)
ax_bar.set_xlabel("Emotion")
ax_bar.set_ylabel("Anzahl")
ax_bar.tick_params(axis="x", rotation=45)
ax_bar.spines[:].set_visible(False)
_ = ax_bar.bar_label(bars, padding=5, fontsize=8)

wedges, _ = ax_pie.pie(values, colors=colors[:len(labels)], startangle=140)
ax_pie.set_title("Anteil der Emotionsklassen", fontsize=13)
ax_pie.legend(wedges, pct_labels, loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9)

plt.tight_layout()
plt.show()


# Step 3: Data Preperation

## DATA CLEANING

| Schritt | Regex-Pattern | Beschreibung |
|---------|---------------|--------------|
| Kleinschreibung | `.lower()` | Vereinheitlichung Groß-/Kleinschreibung |
| URLs entfernen | `https?://\S+\|www\.\S+` | HTTP/HTTPS-Links und www-Adressen |
| Mentions entfernen | `@\w+` | Twitter-Nutzernamen wie `@user` |
| Hashtags entfernen | `#\w+` | Hashtags wie `#happy` |
| Sonderzeichen entfernen | `[^\w\s]` | Punkte, Kommas, Ausrufezeichen etc. |
| Zahlen entfernen | `\d+` | Reine Ziffernfolgen |
| Leerzeichen normalisieren | `\s+` → `' '` | Mehrfache Leerzeichen auf eines reduzieren |

- **Klassenungleichgewicht**: `neutral` (21,6 %) und `worry` (21,1 %) dominieren; `anger` hat nur 110 Einträge (0,3 %).
- *Die **Textlänge** variiert kaum zwischen den Emotionsklassen und ist daher kein Merkmal der Emotion.*
- Für das spätere Modell muss dieses Ungleichgewicht berücksichtigt

Normalisierung, Skalierung und Standardisierung ist für unseren Datensatz nicht relevant, da es sich um textuelle Daten handelt, welche im folgenden sowieso durch Tokenisierung, Vektorisierung etc. weiterverarbeitet werden müssen. Eine verzeitige Normalisierung ist daher nicht notwendig. Normalisierung im Sinne von Lemmatisierung könnte angewendet werden.

In [ ]:
tokens = df.content.str.lower()                                         # Make text small case
tokens = tokens.str.replace(r'https?://\S+|www\.\S+', '', regex=True)   # Remove hyperlinks
tokens = tokens.str.replace(r'@\w+', '', regex=True)                    # Remove user mentions
# tokens = tokens.str.replace(r'#\w+', '', regex=True)                    # Remove hashtags
tokens = tokens.str.replace(r'[^\w\s]', '', regex=True)                 # Remove special characters
tokens = tokens.str.replace(r'\d+', '', regex=True)                     # Remove digits/numbers
tokens = tokens.str.replace(r'\s+', ' ', regex=True).str.strip()        # Remove double spaces
df["tokens"] = tokens

In [ ]:
df = df[df.tokens != ""].dropna()           # Entfernen von leeren Nachrichten (nach dem Cleaning)
df = df.drop_duplicates(subset="tokens")    # Entfernen von Duplikaten (nach dem Cleaning)

In [ ]:
df["tokens"] = df["tokens"].str.split()     # Whitespace Tokenisierung in Spalte eigene Spalte

In [ ]:
df.info()
df.tail()

## 4.1. Datenqualität

- Fokussierung auf Top5 Klassen

In [ ]:
keep = ["neutral", "worry", "happiness", "sadness", "love"]     # Filter auf Top5 Emotionsklassen
df_filtered = df[df.emotion.isin(keep)]

print(f"Verbleibende Klassen: {keep}")
print(f"Anteil am Gesamtdatensatz: {len(df_filtered) / 40000 * 100:.1f}%")
print(f"Verbleibende Einträge: {len(df_filtered)}")

df = df_filtered

In [ ]:
df.info()

## 5.2. Wortfrequenzanalyse

In [ ]:
def plot_top_words(top_n=25):
    word_list = np.concatenate(df.tokens.values)

    word_counts = pd.Series(word_list).value_counts()
    word_counts = word_counts.reset_index()
    word_counts.columns = ["word", "count"]

    print("Anzahl Worte (insgesamt): ", len(word_list))
    print("Anzahl Worte (unique) - |V|=", len(word_counts.word))

    top_words = word_counts.head(top_n)

    _, ax = plt.subplots(figsize=(11, 5))
    bars = ax.bar(top_words["word"], top_words["count"], color="steelblue")
    ax.set_title(f"Top {top_n} häufigste Wörter", fontsize=14)
    ax.set_xlabel("Wort")
    ax.set_ylabel("Anzahl")
    ax.tick_params(axis="x", rotation=45)
    ax.spines[:].set_visible(False)
    _ = ax.bar_label(bars, padding=5, fontsize=8)
    plt.show()

plot_top_words()

Beschreibung der Wortverteilung, Visualisierung

*Ist das hilfreich für unsere Aufgabe?*

### Stopwords entfernen

In [ ]:
stop_words = set(stopwords.words("english"))

print("Anzahl von Stopwords: " + str(len(stop_words)))
print("Stopwords: " + str(stop_words))

In [ ]:
df.tokens = df.tokens.apply(lambda tokens: [w for w in tokens if w not in stop_words])  # Stopwords entfernen

In [ ]:
plot_top_words()

### Lemmatisierung

In [ ]:
df["tokens"] = df.tokens.apply(lambda tokens: [PorterStemmer().stem(w) for w in tokens])

In [ ]:
def top_words_all(n=30):
    return pd.DataFrame([{
        "emotion": emotion,
        "top_words": pd.Series(
            np.concatenate(df[df["emotion"] == emotion]["tokens"].values)
        ).value_counts().head(n).index.tolist()
    } for emotion in df["emotion"].unique()]).set_index("emotion")

top_words_all()

In [ ]:
# Entfernen von Wörtern die in allen Klassen unter den Top30 Wörtern sind

while common := set.intersection(*[set(w) for w in top_words_all().top_words]):
    df.tokens = df.tokens.apply(lambda t: [w for w in t if w not in common])
    print("Removed:", common)

In [ ]:
plot_top_words()

## BYTE PAIR ENCODING

TODO: *Byte Pair Encoding anwenden*

Byte Pair Encoding (BPE) ist für uns nicht weiter relevant, da es nicht auf Wort sondern auf Zeicheneben arbeitet. Dabei werden Wörter in Wortfragmente zerteilt, welches insbesondere für Transformer Modelle wichtig ist. Für einfache Classifier ist dies nicht notwendig oder ggf. sogar kontroproduktiv.

## 5.3. Vektorisierung

In [ ]:
X = df_filtered.tokens.apply(" ".join)
y = df_filtered.emotion

### Bag-of-Words (CountVecotorizer)

In [ ]:
count_vectorizer = CountVectorizer(ngram_range=(1, 2))
X_count = count_vectorizer.fit_transform(X)

print("Anzahl der Features: " + str(count_vectorizer.get_feature_names_out().shape[0]))
print(X_count)

### TF-IDF Ansatz (TfidfVectorizer)

In [ ]:
tfidf_vectorizer = TfidfVectorizer(ngram_range=(2,3))
X_tfidf = tfidf_vectorizer.fit_transform(X)

print("Anzahl der Features: " + str(tfidf_vectorizer.get_feature_names_out().shape[0]))
print(X_tfidf)

### Bigrams

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Top 10 CountVectorizer features by total count
count_sums = np.asarray(X_count.sum(axis=0)).flatten()
count_top10_idx = count_sums.argsort()[::-1][:20]
count_features = count_vectorizer.get_feature_names_out()[count_top10_idx]
count_values = count_sums[count_top10_idx]

# Top 10 TF-IDF features by mean score
tfidf_means = np.asarray(X_tfidf.mean(axis=0)).flatten()
tfidf_top10_idx = tfidf_means.argsort()[::-1][:20]
tfidf_features = tfidf_vectorizer.get_feature_names_out()[tfidf_top10_idx]
tfidf_values = tfidf_means[tfidf_top10_idx]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

bars1 = ax1.barh(count_features[::-1], count_values[::-1], color="steelblue")
ax1.set_title("Top 10 Bigrams – CountVectorizer\n(Gesamthäufigkeit im Trainingsset)", fontsize=12)
ax1.set_xlabel("Summe der Häufigkeiten")
ax1.bar_label(bars1, fmt="{:.0f}", padding=4, fontsize=8)
ax1.spines[:].set_visible(False)

bars2 = ax2.barh(tfidf_features[::-1], tfidf_values[::-1], color="darkorange")
ax2.set_title("Top 10 Bigrams – TF-IDF\n(Mittlerer TF-IDF-Score im Trainingsset)", fontsize=12)
ax2.set_xlabel("Mittlerer TF-IDF-Score")
ax2.bar_label(bars2, fmt="{:.4f}", padding=4, fontsize=8)
ax2.spines[:].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
classes = sorted(y.unique())
count_names = count_vectorizer.get_feature_names_out()
tfidf_names = tfidf_vectorizer.get_feature_names_out()
y_train_arr = y.values

fig, axes = plt.subplots(len(classes), 2, figsize=(16, len(classes) * 3))
fig.suptitle("Top 10 Features pro Klasse", fontsize=14, fontweight="bold", y=1.01)

for row, cls in enumerate(classes):
    mask = y_train_arr == cls

    cls_count = np.asarray(X_count[mask].sum(axis=0)).flatten()
    top_idx = cls_count.argsort()[::-1][:10]
    ax = axes[row, 0]
    bars = ax.barh(count_names[top_idx][::-1], cls_count[top_idx][::-1], color="steelblue")
    ax.set_title(f"{cls} – CountVectorizer", fontsize=10)
    ax.bar_label(bars, fmt="{:.0f}", padding=3, fontsize=7)
    ax.spines[:].set_visible(False)
    ax.tick_params(labelsize=8)

    cls_tfidf = np.asarray(X_tfidf[mask].mean(axis=0)).flatten()
    top_idx = cls_tfidf.argsort()[::-1][:10]
    ax = axes[row, 1]
    bars = ax.barh(tfidf_names[top_idx][::-1], cls_tfidf[top_idx][::-1], color="darkorange")
    ax.set_title(f"{cls} – TF-IDF", fontsize=10)
    ax.bar_label(bars, fmt="{:.4f}", padding=3, fontsize=7)
    ax.spines[:].set_visible(False)
    ax.tick_params(labelsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Mean TF-IDF score per class for every term
class_scores = np.vstack([
    np.asarray(X_tfidf[y_train_arr == cls].mean(axis=0)).flatten()
    for cls in classes
])  # shape: (n_classes, n_features)

# Distinctiveness: score in this class minus max score in any other class
unique_words = {}
for i, cls in enumerate(classes):
    other_max = np.delete(class_scores, i, axis=0).max(axis=0)
    distinctiveness = class_scores[i] - other_max
    top_idx = distinctiveness.argsort()[::-1][:10]
    unique_words[cls] = tfidf_names[top_idx].tolist()

pd.DataFrame(unique_words)

# Step 4: Modellierung

## 6.0 Machine Learning

TODO:

+ Inputvariablen:
+ Outputvariablen: 
+ Prediction vs. Inference: Prediction
+ Classification vs. Regression: Classification

## 7.1 Training und Test von Modellen

In [ ]:
X_count_train, X_count_test, y_count_train, y_count_test = train_test_split(X_count, y, test_size=0.3, stratify=y, random_state=42)
X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test = train_test_split(X_tfidf, y, test_size=0.3, stratify=y, random_state=42)

### K Nearest Neighbor

In [ ]:
# CV + KNN

model_count_knn = KNeighborsClassifier(n_neighbors=21).fit(X_count_train, y_count_train)
print("=== CountVectorizer + KNN ===")
print(classification_report(y_count_test, model_count_knn.predict(X_count_test), zero_division=0))

In [ ]:
# TC + KNN

model_tfidf_knn = KNeighborsClassifier(n_neighbors=21).fit(X_tfidf_train, y_tfidf_train)
print("=== TF-IDF + KNN ===")
print(classification_report(y_tfidf_test, model_tfidf_knn.predict(X_tfidf_test), zero_division=0))

### Random forest

In [ ]:
# CV + RF

model_count_rf = RandomForestClassifier(class_weight="balanced", random_state=42).fit(X_count_train, y_count_train)
print("=== CountVectorizer + RandomForest ===")
print(classification_report(y_count_test, model_count_rf.predict(X_count_test), zero_division=0))

In [ ]:
# TC + RF

model_tfidf_rf = RandomForestClassifier(class_weight="balanced", random_state=42).fit(X_tfidf_train, y_tfidf_train)
print("=== TF-IDF + RandomForest ===")
print(classification_report(y_tfidf_test, model_tfidf_rf.predict(X_tfidf_test), zero_division=0))

# Step 5: Evaluation